<font size="6" color='grey'> <b>
Generative KI. Verstehen. Anwenden. Gestalten.
</b></font> </br>

---

<font size="5" color='grey'> <b>
M05a - Übung A2: Sentiment-Analyse von Produktbewertungen
</b></font> </br>

**Lernziel:** Entwickeln Sie ein Python-Programm für Zero-Shot-Klassifizierung mit aspekt-basierter Sentiment-Analyse von Produktbewertungen.

## Setup & Environment

Umgebung vorbereiten und erforderliche Module laden.

In [ ]:
#@title 🔧 Umgebung einrichten (LOCAL VERSION)
# LOKAL: genai_lib muss bereits installiert sein

import subprocess
import sys

# Erforderliche Packages sicherstellen
required_packages = {
    'dotenv': 'python-dotenv',
    'pandas': 'pandas',
    'tabulate': 'tabulate'
}

for module, package in required_packages.items():
    try:
        __import__(module)
    except ImportError:
        print(f"📦 Installiere {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

from dotenv import load_dotenv
import os

# API Keys aus .env laden
env_path = '/Users/wagnerg/Development/playground/GenAI_GW/.env'
load_dotenv(env_path)

# Imports
from genai_lib.utilities import check_environment, mprint

print("✅ Umgebung wird vorbereitet...")
print()
check_environment()
print()
print(f"✓ OPENAI_API_KEY gesetzt: {'OPENAI_API_KEY' in os.environ and os.environ['OPENAI_API_KEY'] != ''}")
print(f"✓ genai_lib importiert erfolgreich")
print(f"✓ pandas verfügbar")

## Imports

Erforderliche LangChain-Komponenten importieren.

In [ ]:
# Importe für Sentiment-Analyse
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers.string import StrOutputParser
from pandas import DataFrame
import json
import re

## Model-Konfiguration

Parameter und Modell-Initialisierung.

In [ ]:
# Parameter
model_provider = "openai"
model_name = "gpt-4o-mini"
temperature = 0.3

# Modell definieren
llm = init_chat_model(model_name, model_provider=model_provider, temperature=temperature)

print(f"✅ Modell initialisiert: {model_name}")
print(f"   Temperature: {temperature}")

# Sentiment-Analyse von Produktbewertungen

**Aufgabenstellung:**
Entwickeln Sie ein Python-Programm, das Produktbewertungen in die Kategorien "Positiv", "Neutral" oder "Negativ" einordnet und zusätzlich die betroffenen Produktaspekte (Qualität, Preis, Lieferung, Service) identifiziert.

**Lernziele:**
- Zero-Shot-Klassifizierung für mehrere Kategorien implementieren
- Aspekt-basierte Sentiment-Analyse durchführen
- Komplexe Prompts für Mehrfachklassifizierung erstellen
- Strukturierte Ausgaben aus LLM-Antworten generieren

## Beispieldaten

Produktbewertungen für die Analyse.

In [ ]:
# Test-Bewertungen
reviews = [
    "Die Qualität des Produkts ist hervorragend, allerdings finde ich den Preis zu hoch.",
    "Schnelle Lieferung, guter Service, faire Preise - besser geht es nicht!",
    "Nach zwei Wochen ging das Gerät kaputt. Der Kundenservice war bei der Reklamation leider keine Hilfe.",
    "Durchschnittliche Qualität, erfüllt seinen Zweck. Lieferung dauerte etwas länger als angegeben."
]

print(f"📊 {len(reviews)} Bewertungen zur Analyse vorhanden")

## Sentiment-Analyse System

Definiere das Zero-Shot Klassifizierungs-System.

In [ ]:
# System Prompt für Sentiment-Analyse
system_prompt = """
Du bist ein Experte für Sentiment-Analyse von Produktbewertungen.

Analysiere die folgende Produktbewertung hinsichtlich:
1. Gesamtsentiment: Positiv / Neutral / Negativ
2. Erwähnte Produktaspekte: Qualität, Preis, Lieferung, Service
3. Sentiment pro Aspekt: Positiv / Negativ / Nicht erwähnt

Antworte im folgenden Format:
GESAMTSENTIMENT: [Sentiment]
QUALITÄT: [Sentiment]
PREIS: [Sentiment]
LIEFERUNG: [Sentiment]
SERVICE: [Sentiment]
BEGRÜNDUNG: [1-2 Sätze Erklärung]
"""

# Prompt-Template
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "Bewertung: {review}")
])

# Parser und Chain
parser = StrOutputParser()
chain = prompt | llm | parser

print("✅ Sentiment-Analyse-System initialisiert")

## Hilfsfunktion für strukturierte Ausgabe

Parse LLM-Output in strukturierte Daten.

In [ ]:
def parse_sentiment_response(response_text: str) -> dict:
    """
    Parse die LLM-Antwort in strukturierte Daten.
    """
    result = {
        "gesamtsentiment": "Unbekannt",
        "aspekte": {
            "Qualität": "Nicht erwähnt",
            "Preis": "Nicht erwähnt",
            "Lieferung": "Nicht erwähnt",
            "Service": "Nicht erwähnt"
        },
        "begründung": ""
    }
    
    lines = response_text.strip().split('\n')
    
    for line in lines:
        if ':' in line:
            key, value = line.split(':', 1)
            key = key.strip().upper()
            value = value.strip()
            
            if key == "GESAMTSENTIMENT":
                result["gesamtsentiment"] = value
            elif key == "QUALITÄT":
                result["aspekte"]["Qualität"] = value
            elif key == "PREIS":
                result["aspekte"]["Preis"] = value
            elif key == "LIEFERUNG":
                result["aspekte"]["Lieferung"] = value
            elif key == "SERVICE":
                result["aspekte"]["Service"] = value
            elif key == "BEGRÜNDUNG":
                result["begründung"] = value
    
    return result

## Sentiment-Analyse durchführen

Analysiere alle Bewertungen.

In [ ]:
# Analysiere alle Bewertungen
results = []

mprint("## ⭐ Sentiment-Analyse von Produktbewertungen")
mprint("---")

for i, review in enumerate(reviews, 1):
    mprint(f"### Bewertung {i}")
    mprint(f"**Text:** {review}")
    
    # LLM aufrufen
    response = chain.invoke({"review": review})
    
    # Parse Antwort
    parsed = parse_sentiment_response(response)
    
    # Ausgabe
    mprint(f"**Gesamtsentiment:** `{parsed['gesamtsentiment']}`")
    mprint(f"**Aspekte:**")
    for aspekt, sentiment in parsed["aspekte"].items():
        mprint(f"- {aspekt}: `{sentiment}`")
    mprint(f"**Begründung:** {parsed['begründung']}")
    print()
    
    # Speichere Ergebnis
    results.append({
        "Bewertung_ID": i,
        "Text": review[:40] + "...",
        "Sentiment": parsed["gesamtsentiment"],
        **{f"Aspekt_{k}": v for k, v in parsed["aspekte"].items()}
    })

## Zusammenfassung als Tabelle

Zeige alle Ergebnisse in strukturierter Form.

In [ ]:
# Erstelle DataFrame
df_results = DataFrame(results)

mprint("## 📊 Übersicht aller Analysen")
mprint("---")
mprint(df_results.to_markdown(index=False))

## Statistik

Auswertung der Ergebnisse.

In [ ]:
# Statistiken berechnen
mprint("## 📈 Statistiken")
mprint("---")

sentiments = [r["Sentiment"] for r in results]
sentiment_counts = {}
for s in sentiments:
    sentiment_counts[s] = sentiment_counts.get(s, 0) + 1

mprint(f"**Gesamtsentiment-Verteilung:**")
for sentiment, count in sorted(sentiment_counts.items()):
    percentage = (count / len(results)) * 100
    mprint(f"- {sentiment}: {count} ({percentage:.0f}%)")

# Aspekt-Analyse
mprint(f"\n**Häufigste positive Aspekte:**")
aspect_sentiment = {}
for result in results:
    for key, value in result.items():
        if key.startswith("Aspekt_"):
            aspect_name = key.replace("Aspekt_", "")
            if aspect_name not in aspect_sentiment:
                aspect_sentiment[aspect_name] = {"Positiv": 0, "Negativ": 0, "Nicht erwähnt": 0}
            aspect_sentiment[aspect_name][value] = aspect_sentiment[aspect_name].get(value, 0) + 1

for aspect, sentiments_dict in sorted(aspect_sentiment.items()):
    positiv_count = sentiments_dict.get("Positiv", 0)
    total_mentioned = len(results) - sentiments_dict.get("Nicht erwähnt", 0)
    if total_mentioned > 0:
        mprint(f"- {aspect}: {positiv_count}/{total_mentioned} positiv")

## Erweiterte Beispiele

Teste mit zusätzlichen Bewertungen.

In [ ]:
# Zusätzliche Beispiel-Bewertungen
additional_reviews = [
    "Absolut empfehlenswert! Großartige Qualität zu fairen Preisen. Versand war super schnell.",
    "Schlecht verpackt, Produkt kam beschädigt an. Rückgabe war sehr umständlich.",
    "Preis-Leistung ist okay. Nichts Besonderes, aber auch nicht schlecht."
]

mprint("## 🆕 Zusätzliche Bewertungen")
mprint("---")

for i, review in enumerate(additional_reviews, 1):
    mprint(f"### Zusatz-Bewertung {i}")
    mprint(f"**Text:** {review}")
    
    # LLM aufrufen
    response = chain.invoke({"review": review})
    
    # Parse Antwort
    parsed = parse_sentiment_response(response)
    
    # Ausgabe
    mprint(f"**Gesamtsentiment:** `{parsed['gesamtsentiment']}`")
    mprint(f"**Aspekte:**")
    for aspekt, sentiment in parsed["aspekte"].items():
        if sentiment != "Nicht erwähnt":
            mprint(f"- {aspekt}: `{sentiment}`")
    mprint(f"**Begründung:** {parsed['begründung']}")
    print()

## 💡 Erkenntnisse

### Was macht diese Lösung effektiv?

**1. Zero-Shot Klassifizierung**
- Das Modell benötigt keine Trainingsbeispiele
- Funktioniert sofort mit klaren Anweisungen
- Flexibel für neue Szenarien

**2. Aspekt-basierte Analyse**
- Identifiziert mehrere Dimensionen gleichzeitig
- Zeigt, welche Aspekte positiv/negativ bewertet werden
- Hilft bei gezielten Verbesserungen

**3. Strukturierte Ausgabe**
- Leicht zu parsbare Antworten
- Können in Datenbanken gespeichert werden
- Ermöglichen statistische Auswertungen

**4. Kontextuales Verständnis**
- LLMs verstehen Sarkasmus und Kontext
- Können nuancierte Bewertungen erfassen
- Bessere Genauigkeit als regelbasierte Systeme

### Best Practices

✅ **Do:**
- Struktur und Format explizit vorgeben
- Klare Kategorien definieren
- Responses in strukturierte Daten parsen
- Batch-Verarbeitung für mehrere Items nutzen

❌ **Don't:**
- Zu komplexe Kategorien verwenden
- Mehrdeutige Aspekte-Definitionen geben
- Output-Format nicht vorgeben
- Zu hohe Temperature für konsistente Klassifizierung

## 📝 Zusammenfassung

✅ **Was wir gelernt haben:**

1. **Zero-Shot Klassifizierung**: Mit klaren Anweisungen können LLMs ohne Training klassifizieren
2. **Aspekt-basierte Sentiment-Analyse**: Mehrere Dimensionen gleichzeitig analysieren
3. **Strukturierte Prompts**: Format-Vorgaben führen zu konsistenten Ergebnissen
4. **Output Parsing**: LLM-Responses strukturiert extrahieren und verarbeiten
5. **Statistische Auswertung**: Ergebnisse analysieren und visualisieren

🚀 **Erweiterungsmöglichkeiten:**
- Weitere Aspekte hinzufügen (z.B. Verpackung, Garantie)
- Confidence-Score für jede Klassifizierung
- Vergleich mit anderen LLM-Modellen
- Export in verschiedene Formate (CSV, JSON, PDF)
- Echtzeit-Monitoring von Produktbewertungen